# 1. Data preparation

This notebook builds `data/prefectures.csv`, the table that the other notebooks read.
It has one row per prefecture and these columns:

| Column | Contents | Source |
|---|---|---|
| `prefecture`, `code` | prefecture name and its standard code (1 = Hokkaido, ..., 47 = Okinawa), used to draw the maps | |
| `search_interest_<period>` | Search Interest in each of the five periods | `data/search_interest/by_prefecture_<period>.csv` |
| `host_<period>` | 1 if the prefecture had at least one operable reactor in the period, 0 if not | `data/reference/reactor_status.csv` |
| `border_2004-2010` | 1 if the prefecture borders a host prefecture in 2004–2010, 0 if not | prefecture borders from the `japanmap` package |
| `distance_km` | distance from Fukushima Prefecture (Fukushima = 0) | `data/reference/distance_from_fukushima.csv` |

Run this notebook first.

In [1]:
from pathlib import Path
import pandas as pd
from japanmap import adjacent

DATA = Path("../data")
PERIODS = {"2004-2010": (2004, 2010), "2011": (2011, 2011), "2012-2014": (2012, 2014),
           "2015-2019": (2015, 2019), "2020-2024": (2020, 2024)}

## Search Interest

All files list the 47 prefectures in the order of their standard codes, so the code is the row number.

In [2]:
table = pd.read_csv(DATA / "search_interest" / "by_prefecture_2004-2010.csv")[["prefecture"]]
table.insert(1, "code", range(1, 48))
assert table.set_index("code").loc[[1, 7, 13, 47], "prefecture"].tolist() == ["Hokkaido", "Fukushima", "Tokyo", "Okinawa"]

for period in PERIODS:
    d = pd.read_csv(DATA / "search_interest" / f"by_prefecture_{period}.csv")
    assert d["prefecture"].tolist() == table["prefecture"].tolist()
    table[f"search_interest_{period}"] = d["search_interest"]

## Host status

A prefecture is a host in a period if at least one reactor located there was operable (`Y`) in at least one
year of the period.
In `reactor_status.csv`, a reactor is counted as not operable (`N`) from the year after its retirement date
(廃止日) in the JAIF table.

In [3]:
status = pd.read_csv(DATA / "reference" / "reactor_status.csv")
for period, (first, last) in PERIODS.items():
    years = [str(y) for y in range(first, last + 1)]
    hosts = set(status.loc[status[years].eq("Y").any(axis=1), "prefecture"])
    table[f"host_{period}"] = table["prefecture"].isin(hosts).astype(int)
    names = table.loc[table[f"host_{period}"] == 1, "prefecture"].tolist()
    print(f"{period}, {len(names)} hosts: {', '.join(names)}")

2004-2010, 13 hosts: Hokkaido, Aomori, Miyagi, Fukushima, Ibaraki, Niigata, Ishikawa, Fukui, Shizuoka, Shimane, Ehime, Saga, Kagoshima
2011, 13 hosts: Hokkaido, Aomori, Miyagi, Fukushima, Ibaraki, Niigata, Ishikawa, Fukui, Shizuoka, Shimane, Ehime, Saga, Kagoshima
2012-2014, 13 hosts: Hokkaido, Aomori, Miyagi, Fukushima, Ibaraki, Niigata, Ishikawa, Fukui, Shizuoka, Shimane, Ehime, Saga, Kagoshima
2015-2019, 13 hosts: Hokkaido, Aomori, Miyagi, Fukushima, Ibaraki, Niigata, Ishikawa, Fukui, Shizuoka, Shimane, Ehime, Saga, Kagoshima
2020-2024, 12 hosts: Hokkaido, Aomori, Miyagi, Ibaraki, Niigata, Ishikawa, Fukui, Shizuoka, Shimane, Ehime, Saga, Kagoshima


## Border status (2004–2010)

The land borders between prefectures come from the `japanmap` package, so Hokkaido and Okinawa have no neighbors.
A prefecture has a border status of 1 if it borders a prefecture with a host status of 1 (hosting an operable
reactor), as set in the previous cell.
This is performed only for 2004–2010.

In [4]:
is_host = dict(zip(table["code"], table["host_2004-2010"]))
table["border_2004-2010"] = [int(any(is_host[n] for n in adjacent(code))) for code in table["code"]]

## Distance from Fukushima

In [5]:
distance = pd.read_csv(DATA / "reference" / "distance_from_fukushima.csv")
assert distance["prefecture"].tolist() == table["prefecture"].tolist()
table["distance_km"] = distance["distance_km"]

## The table

In [6]:
table.to_csv(DATA / "prefectures.csv", index=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
table

,prefecture,code,search_interest_2004-2010,search_interest_2011,search_interest_2012-2014,search_interest_2015-2019,search_interest_2020-2024,host_2004-2010,host_2011,host_2012-2014,host_2015-2019,host_2020-2024,border_2004-2010,distance_km
0,Hokkaido,1,39,36,34,35,26,1,1,1,1,1,0,594.8
1,Aomori,2,74,32,30,42,31,1,1,1,1,1,0,342.1
2,Iwate,3,45,32,32,29,20,0,0,0,0,0,1,224.9
3,Miyagi,4,71,40,46,55,38,1,1,1,1,1,1,67.7
4,Akita,5,38,34,25,29,19,0,0,0,0,0,1,220.8
5,Yamagata,6,37,40,33,29,25,0,0,0,0,0,1,55.2
6,Fukushima,7,86,100,100,100,100,1,1,1,1,0,1,0.0
7,Ibaraki,8,60,60,45,58,34,1,1,1,1,1,1,156.3
8,Tochigi,9,34,52,30,26,23,0,0,0,0,0,1,141.3
9,Gunma,10,37,44,27,28,20,0,0,0,0,0,1,195.9
